<a href="https://colab.research.google.com/github/ANKITHATALAPALLY/ANKITHA-T_4NI23CI017_MLLAB_E1/blob/main/MLLAB10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# Install SHAP
!pip install -q shap

# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


# Load dataset
data = load_breast_cancer()

X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

print("Dataset loaded successfully!")
print("Dataset shape:", X.shape)

display(X.head())

print("\nTarget classes:", data.target_names)

print("\nClass distribution:")
print(pd.Series(y).value_counts())


# Split dataset into training and testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


# Create Random Forest model
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)


# Train model
model.fit(X_train, y_train)


# Make predictions
y_pred = model.predict(X_test)


# Evaluate model
accuracy = accuracy_score(y_test, y_pred)

print("Model trained successfully!")
print("Accuracy:", round(accuracy * 100, 2), "%")

print("\nClassification Report:")

print(classification_report(
    y_test,
    y_pred,
    target_names=data.target_names
))


# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 4))
plt.imshow(cm, cmap='Blues')
plt.colorbar()

plt.xticks([0, 1], data.target_names)
plt.yticks([0, 1], data.target_names)

for i in range(2):
    for j in range(2):
        plt.text(
            j, i, cm[i, j],
            ha='center',
            va='center'
        )

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()


# Create SHAP Tree Explainer
explainer = shap.TreeExplainer(model)


# Calculate SHAP values for test data
shap_values = explainer.shap_values(X_test)


# Select Malignant class (class 0)
if isinstance(shap_values, list):
    malignant_shap = shap_values[0]

elif np.asarray(shap_values).ndim == 3:
    malignant_shap = np.asarray(shap_values)[:, :, 0]

else:
    malignant_shap = np.asarray(shap_values)


# Global feature importance
importance = np.abs(malignant_shap).mean(axis=0)

importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': importance
}).sort_values(
    by='Importance',
    ascending=False
)

print("Top 10 Important Features:")
display(importance_df.head(10))


# Plot feature importance
plt.figure(figsize=(10, 6))

plt.barh(
    importance_df['Feature'].head(10)[::-1],
    importance_df['Importance'].head(10)[::-1]
)

plt.xlabel("Mean Absolute SHAP Value")
plt.title("Global Feature Importance")
plt.show()

plt.close()


# SHAP Summary Plot
shap.summary_plot(
    malignant_shap,
    X_test,
    show=False
)

plt.title("SHAP Summary Plot")
plt.tight_layout()
plt.show()

plt.close()


# Explain two individual test instances
for i in range(2):

    print("\n" + "="*40)

    print("Explaining Test Instance:", i + 1)

    print(
        "Actual Diagnosis:",
        data.target_names[y_test[i]]
    )

    print(
        "Predicted Diagnosis:",
        data.target_names[y_pred[i]]
    )

    # Probability prediction
    test_instance = X_test.iloc[[i]]

    prob_malignant = model.predict_proba(
        test_instance
    )[0][0]

    print(
        "Probability of Malignant:",
        round(prob_malignant * 100, 2),
        "%"
    )

    # Expected value for malignant class
    expected = np.asarray(
        explainer.expected_value
    ).flatten()[0]

    # Create SHAP explanation object
    explanation = shap.Explanation(
        values=malignant_shap[i],
        base_values=expected,
        data=X_test.iloc[i].values,
        feature_names=X.columns.tolist()
    )

    # Display individual feature contributions
    plt.figure()

    shap.plots.waterfall(
        explanation,
        max_display=10,
        show=False
    )

    plt.title(
        f"Waterfall Plot - Instance {i+1}",
        fontsize=12
    )

    plt.tight_layout()
    plt.show()
    plt.close()
